## Setup start 

In [ ]:
source("~/workspace/pipelines/snt_dhis2_formatting/utils/snt_dhis2_formatting.r")
snt_paths <- init_snt_workspace(
    snt_pipeline_name="snt_dhis2_formatting",
    packages=c("arrow", "dplyr", "tidyr", "stringr", "stringi", "jsonlite", "httr", "glue"))

# Load config
config_json <- load_snt_config(file.path(snt_paths$CONFIG_PATH, "SNT_config.json"))

# Save config variables
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
ADMIN_1 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_1)
ADMIN_2 <- toupper(config_json$SNT_CONFIG$DHIS2_ADMINISTRATION_2)
extracts_dataset_id <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_EXTRACTS
ou_level <- config_json$SNT_CONFIG$ANALYTICS_ORG_UNITS_LEVEL

### Load DHIS2 reporting rates data  

-Load DHIS2 reporting rates data from latest dataset version (extracts)

In [ ]:
# Load file from dataset
dhis2_data <- load_dataset_file(extracts_dataset_id, paste0(COUNTRY_CODE, "_dhis2_raw_reporting.parquet"), verbose=FALSE)
log_msg(glue("DHIS2 reporting rate data loaded from dataset : '{extracts_dataset_id}' dataframe dimensions: {paste(dim(dhis2_data), collapse=', ')}"))
head(dhis2_data, 3)

## Reporting rates dataset formatting

### Format (clean) pyramid and dataset names

In [ ]:
log_msg(glue("Start DHIS2 reporting rates formatting."))   

# Standard string formats 
dhis2_data_clean <- clean_input_data(dhis2_data, verbose=FALSE)
head(dhis2_data_clean, 3)

### Column selection

In [ ]:
# Set administrative columns
adm_1_id_col <- gsub("_NAME", "_ID", ADMIN_1)
adm_1_name_col <- ADMIN_1
adm_2_id_col <- gsub("_NAME", "_ID", ADMIN_2)
adm_2_name_col <- ADMIN_2
adm_ou_id_col <- glue("LEVEL_{ou_level}_ID")
adm_ou_name_col <- glue("LEVEL_{ou_level}_NAME")

# Administrative columns list
admin_columns <- c(
    adm_1_id_col,
    adm_1_name_col,
    adm_2_id_col,
    adm_2_name_col,
    adm_ou_id_col,
    adm_ou_name_col
) 

# Reporting rates based on indicators do not have data to the OU level, data is retrieved aggregated at ADMIN 2.
if (!(adm_ou_id_col %in% colnames(dhis2_data_clean)) | !(adm_ou_name_col %in% colnames(dhis2_data_clean))) {
    log_msg("'OU' column names not found in reporting data, running for reporting rates indicators.", "warning")
}

# Select relevant columns for SNT
fixed_cols <- c("PE", "VALUE", "PRODUCT_UID", "PRODUCT_NAME", "PRODUCT_METRIC")
selected_cols <-  c(fixed_cols, admin_columns)
dhis2_data_selection <- dhis2_data_clean %>% select(any_of(selected_cols))

print(dim(dhis2_data_selection))
head(dhis2_data_selection, 3)

### Apply SNT format 

In [ ]:
# Select and rename columns
dhis2_data_formatted <- dhis2_data_selection %>%
    mutate(
        PE = as.numeric(PE),
        YEAR = as.numeric(substr(PE, 1, 4)),
        MONTH = as.numeric(substr(PE, 5, 6)),
        VALUE = as.numeric(VALUE)
    ) %>%
    select(
        PERIOD = PE,
        YEAR,
        MONTH,
        ADM1_NAME = !!sym(adm_1_name_col),
        ADM1_ID = !!sym(adm_1_id_col),           
        ADM2_NAME = !!sym(adm_2_name_col),
        ADM2_ID = !!sym(adm_2_id_col),          
        any_of(c(OU_ID = adm_ou_id_col, OU_NAME = adm_ou_name_col)),
        all_of(fixed_cols)
     )

# Sort dataframe by period
dhis2_data_formatted <- dhis2_data_formatted[order(as.numeric(dhis2_data_formatted$PERIOD)), ]

print(dim(dhis2_data_formatted))
head(dhis2_data_formatted, 3)

## Output formatted population data

In [ ]:
FORMATTED_DATA_PATH <- file.path(snt_paths$DATA_PATH, "dhis2", "extracts_formatted")

# write files
write_parquet(dhis2_data_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_reporting.parquet")))
write.csv(dhis2_data_formatted, file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, "_reporting.csv")), row.names = FALSE)

# log
log_msg(glue("Formatted reporting data saved under: {file.path(FORMATTED_DATA_PATH, paste0(COUNTRY_CODE, '_reporting.parquet'))}"))

### Data Summary 

In [ ]:
# Data summary
print(summary(dhis2_data_formatted))